In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from google.colab import drive

# 1. 구글 드라이브 연결
drive.mount('/content/drive')

# 2. 경로 설정
zip_path = '/content/drive/MyDrive/fine_tuning_dataset_v2.zip'
target_dir = '/content/fine_tuning_dataset_v2' # 압축을 풀 코랩 로컬 폴더

# 폴더가 없으면 생성
os.makedirs(target_dir, exist_ok=True)

# 3. 리눅스 unzip 명령어로 초고속 압축 해제
# -q: 파일 목록 출력 생략 (속도 대폭 향상 및 브라우저 멈춤 방지)
# -o: 덮어쓰기 허용
# -d: 저장할 목적지 폴더 지정
print("구글 드라이브에서 코랩 로컬로 압축 해제 중")
!unzip -q -o {zip_path} -d {target_dir}
print("압축 해제 완료")

# 4. 검증: 파일이 제대로 풀렸는지 폴더별 개수 확인
print("\n 데이터셋 폴더 현황:")
for folder in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    folder_path = os.path.join(target_dir, folder)
    if os.path.exists(folder_path):
        print(f"{folder}: {len(os.listdir(folder_path))}개")
    else:
        print(f"{folder} 폴더를 찾을 수 없습니다.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ 구글 드라이브에서 코랩 로컬로 압축 해제 중... (약 1~3분 소요 예상)
✅ 압축 해제 완료!

📊 데이터셋 폴더 현황:
  ⚠️ images/train 폴더를 찾을 수 없습니다.
  ⚠️ images/val 폴더를 찾을 수 없습니다.
  ⚠️ labels/train 폴더를 찾을 수 없습니다.
  ⚠️ labels/val 폴더를 찾을 수 없습니다.


In [ ]:
import os
import glob

# 하위 폴더까지 모두 검색하기 위해 recursive=True와 ** 사용
train_labels = glob.glob('/content/fine_tuning_dataset_v2/train/**/*.txt', recursive=True)
valid_labels = glob.glob('/content/fine_tuning_dataset_v2/**/*.txt', recursive=True)
all_labels = train_labels + valid_labels

class_counts = {0: 0, 1: 0}
total_boxes = 0

print(f"총 {len(all_labels)}개의 라벨 파일 분석 시작...")

for label_path in all_labels:
    try:
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts: continue

                class_id = int(parts[0])
                if class_id in class_counts:
                    class_counts[class_id] += 1
                    total_boxes += 1
    except Exception as e:
        print(f"Error reading {label_path}: {e}")

print("=" * 50)
print(f"[라벨 전수조사 결과] 총 라벨 파일 수: {len(all_labels)}개")
print(f"전체 Bounding Box 개수: {total_boxes}개")
print("-" * 50)
print(f"Class 0 (YES_Helmet) 개수 : {class_counts[0]:,}개")
print(f"Class 1 (NO_Helmet) 개수   : {class_counts[1]:,}개")
print("=" * 50)

# 추가 팁: 클래스 분포 비율 확인
if total_boxes > 0:
    ratio0 = (class_counts[0] / total_boxes) * 100
    ratio1 = (class_counts[1] / total_boxes) * 100
    print(f"분포 비율 -> YES: {ratio0:.1f}%, NO: {ratio1:.1f}%")

🚀 총 73847개의 라벨 파일 분석 시작...
📈 [라벨 전수조사 결과] 총 라벨 파일 수: 73847개
📦 전체 Bounding Box 개수: 126646개
--------------------------------------------------
🪖 Class 0 (YES_Helmet) 개수 : 103,321개
🧑 Class 1 (NO_Helmet) 개수   : 23,325개
분포 비율 -> YES: 81.6%, NO: 18.4%


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.4 MB/s eta 0:00:00


In [ ]:
%%writefile /content/fine_tuning_dataset_v2/data.yaml
train: /content/fine_tuning_dataset_v2/images/train
val: /content/fine_tuning_dataset_v2/images/val

# 클래스 개수
nc: 2

# 클래스 이름 (0번: YES_Helmet, 1번: NO_Helmet)
names:
  0: YES_Helmet
  1: NO_Helmet

Writing /content/fine_tuning_dataset_v2/data.yaml


In [ ]:
from ultralytics import YOLO

model = YOLO('best.pt')

results = model.train(
    data='/content/fine_tuning_dataset_v2/data.yaml',
    optimizer='MuSGD',
    epochs=50,
    imgsz=640,
    batch=128,
    workers=8,
    lr0=0.02,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    freeze=0,
    cls=2.5,                   # 클래스 분류 오류에 강한 페널티 부여
    save_period=5,
    mosaic=1.0,                # 4장 이미지를 하나로 합쳐 배경 다양성 확보
    mixup=0.1                  # 이미지 겹치기로 객체 인식력 강화
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fine_tuning_dataset_v2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, 

In [ ]:
from ultralytics import YOLO
import cv2

model = YOLO('/content/runs/detect/train/weights/epoch45.pt')

# 영상 경로 설정
video_path = '/content/Take Time to Take Care (Vehicular Safety).mp4'

# 추론 실행 및 결과 저장
# conf=0.3: 신뢰도 임계값, 원하시는 수치로 조정 가능합니다.
# save=True: 결과를 영상으로 저장합니다.
results = model.predict(
    source=video_path,
    conf=0.3,
    save=True,
    imgsz=640,
    device=0,           # GPU 사용
    stream=True         # 대용량 영상 처리를 위한 스트리밍 방식
)

# 결과 출력
# stream=True를 사용하면 generator 객체가 반환되므로 루프를 돌려야 합니다.
for r in results:
    # 각 프레임별로 추론 결과를 처리하거나 시각화할 수 있습니다.
    # r.plot()을 사용하면 바운딩 박스가 그려진 이미지가 생성됩니다.
    annotated_frame = r.plot()

    # 화면 표시를 원하시면 cv2.imshow를 쓰지만, 코랩에서는
    # 자동으로 save=True 설정에 의해 파일로 저장됩니다.
    pass

print("영상 추론 및 결과 저장 완료!")


video 1/1 (frame 1/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 14.1ms
video 1/1 (frame 2/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 12.7ms
video 1/1 (frame 3/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 12.1ms
video 1/1 (frame 4/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 12.6ms
video 1/1 (frame 5/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 12.5ms
video 1/1 (frame 6/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 13.6ms
video 1/1 (frame 7/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 12.8ms
video 1/1 (frame 8/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 14.1ms
video 1/1 (frame 9/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 YES_Helmet, 13.0ms
video 1/1